<a href="https://colab.research.google.com/github/njones61/xslope/blob/main/notebooks/xslope_lem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XSLOPE - Limit Equilibrium Method

This notebook illustrates how to use xslope to solve basic slope stability problems using the limit equilibrium method.

## Install xslope and import functions

In [ ]:
# Install the xslope package
# - Basic install (limit equilibrium only): pip install xslope
# - Full install (with FEM features): pip install xslope[fem]

%%capture
!pip install xslope

In [ ]:
# Import functions

import xslope as xslope
print(f"xslope version: {xslope.__version__}") 

from xslope.global_config import non_circ
from xslope.slice import generate_slices
from xslope.fileio import load_slope_data, load_data_from_pickle
from xslope.plot import plot_circular_search_results, plot_inputs, plot_solution, plot_noncircular_search_results, plot_reliability_results
from xslope.solve import solve_selected, solve_all
from xslope.search import circular_search, noncircular_search
from xslope.summary import print_ito_matsui_summary, print_rapid_drawdown_summary, print_no_solution_warning
from xslope.advanced import reliability as reliability_analysis

## Upload Excel Template

In [ ]:
# Upload excel input template for selected problem

from google.colab import files
upload = files.upload()
file_name = list(upload.keys())[0]

# See if uploaded file is a zip archive. If so, unzip it
if file_name.endswith('.zip'):
  import zipfile
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
    # Extract all files
    zip_ref.extractall()
    extracted_files = zip_ref.namelist()

    # Find the excel file in the extracted list
    excel_file_found = False
    for f in extracted_files:
      if f.endswith('.xlsx'):
        file_name = f
        excel_file_found = True
        print(f"Found Excel file: {file_name}")
        break

    if not excel_file_found:
        # If no excel file is found at all, print an error and set file_name to None
        print("Error: No .xlsx file found in the uploaded archive. Please ensure your zip file contains an .xlsx file.")
        file_name = None

## Load slope data

In [ ]:
slope_data = load_slope_data(file_name)
plot_inputs(slope_data, mode='lem', save_png=False)

In [ ]:
# @title Select Options {"run":"auto"}
method = "spencer" # @param ["oms","bishop","janbu","corps_engineers","lowe_karafiath","spencer","morgenstern_price"]
num_slices = 30 # @param {"type":"integer"}
analysis_type = "auto_search" # @param ["single_surface","auto_search","reliability"]
surface_type = "circular" # @param ["circular","non_circular"]
rapid_drawdown = False # @param {"type":"boolean"}
save_png = True # @param {"type":"boolean","placeholder":"True"}
diagnostic = False # @param {"type":"boolean"}

## Find and plot solution

In [ ]:
if analysis_type == 'single_surface': # analyze the specified failure surface
  circle = slope_data['circles'][0] if slope_data['circular'] else None
  non_circ = slope_data['non_circ'] if slope_data['non_circ'] else None
  success, result = generate_slices(slope_data, circle=circle, non_circ=non_circ, num_slices=num_slices)
  if success:
      slice_df, failure_surface = result
      results = solve_selected(method, slice_df, rapid=rapid_drawdown)
      if isinstance(results, dict):
          plot_solution(slope_data, slice_df, failure_surface, results, save_png=save_png)
      else:
          print(f"No solution to plot.")
  else:
      print(result)
      exit()

elif analysis_type == "auto_search": # automated search for critical surface
  if surface_type == "circular":
    fs_cache, converged, search_path, circle_cache = circular_search(slope_data, method, rapid=rapid_drawdown, num_slices=num_slices, diagnostic=diagnostic)
    plot_circular_search_results(slope_data, fs_cache, search_path, circle_cache=circle_cache, save_png=save_png)
  else:
    fs_cache, converged, search_path = noncircular_search(slope_data, method, rapid=rapid_drawdown, diagnostic=diagnostic)
    plot_noncircular_search_results(slope_data, fs_cache, search_path, save_png=save_png)

  # Extract critical failure surface (lowest FS is first in sorted list)
  critical_surface = fs_cache[0]
  slice_df = critical_surface['slices']
  failure_surface = critical_surface['failure_surface']
  results = critical_surface['solver_result']
  print_ito_matsui_summary(slope_data, slice_df)
  if rapid_drawdown:
    print_rapid_drawdown_summary(results)
  if results is None:
    print_no_solution_warning()
  else:
    plot_solution(slope_data, slice_df, failure_surface, results, save_png=save_png)

elif analysis_type == "reliability": # reliability analysis (supports both circular and non-circular)
  circular = (surface_type == "circular")
  success, result = reliability_analysis(slope_data, method, rapid=rapid_drawdown, circular=circular, debug_level=1)
  if success:
    plot_reliability_results(slope_data, result, save_png=save_png)
  else:
    print(f"Reliability analysis failed: {result}")